In [68]:
import pandas as pd
import mlflow
import subprocess
import re
import os
from pathlib import Path
import time
from datetime import datetime
import sys


# GetPythonPath

In [69]:
print(sys.executable)
print("Current Time:", datetime.now().strftime("%H:%M:%S"))

c:\Users\peter\anaconda3\python.exe
Current Time: 23:38:44


# Save

In [ ]:

########################################################
# Setup
########################################################

#ProjectFolder
os.chdir(r"C:\GitTestOnC\Test4_Oct2\Git")

runName = "Test06_Oct2_11.38___SmDta"
notebookName = "main.ipynb"

mult = 2
add = 4

In [ ]:
########################################################
# Open Data
########################################################

df = pd.read_csv("data.csv", header=None)

########################################################
# Calc
########################################################

df_after = add + df * mult
sumResult = df_after.to_numpy().sum()

In [ ]:
df

,0,1,2
0,1,2,3
1,4,5,6
2,7,8,9
3,10,11,12


In [ ]:
df_after

,0,1,2
0,6,8,10
1,12,14,16
2,18,20,22
3,24,26,28


In [ ]:
sumResult

np.int64(204)

In [ ]:
########################################################
# Setup MlFlow
########################################################

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(Path(notebookName).stem)

########################################################
# Save Data Version With DVC
########################################################

subprocess.run(["py", "-m", "dvc", "add", "data.csv"], check=True)
subprocess.run(["py", "-m", "dvc", "push"], check=True)

dvcText = Path("data.csv.dvc").read_text()
dvcHash = re.search(r"md5:\s*([0-9a-f]+)", dvcText).group(1)

########################################################
# Save Code + DVC Pointer To Git
########################################################

subprocess.run(["git", "add", notebookName, "data.csv.dvc"], check=True)

hasChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasChanges:
    subprocess.run(["git", "commit", "-m", f"Code for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Get Git Commit
########################################################

gitCommit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

########################################################
# Save To MlFlow
########################################################

with mlflow.start_run(run_name=runName) as run:
    runId = run.info.run_id
    mlflow.log_params({"mult": mult, "add": add})
    mlflow.log_metric("sum", sumResult)
    mlflow.set_tag("git_commit", gitCommit)
    mlflow.set_tag("dvc_data_hash", dvcHash)
    mlflow.log_artifact("data.csv.dvc", artifact_path="data")

########################################################
# Print
########################################################

print("Git commit =", gitCommit)
print("DVC hash =", dvcHash)
print("MlFlow experiment =", Path(notebookName).stem)
print("MlFlow run =", runName)
print("MlFlow run ID =", runId)
#PrintTime
print("Current Time:", datetime.now().strftime("%H:%M:%S"))


DVC hash = 2e53a8406cd074ffd764730c32e3cd00
Git commit = 30ea4440aa6b11055b81a9d993ac7d702917b46d
🏃 View run Test06_Oct2_11.17___SmDta at: http://127.0.0.1:5000/#/experiments/2/runs/1b54da18dce94e05a7116e4974ba53d7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


CompletedProcess(args=['git', 'push', 'origin', 'Test06_Oct2_11.17___SmDta'], returncode=0)

In [ ]:
########################################################
# Export Notebook To HTML
########################################################

Path("results").mkdir(exist_ok=True)
htmlFile = Path(f"results/{runName}.html")

#DeleteOldHtml
if htmlFile.exists(): htmlFile.unlink()

#ExportHtml
subprocess.run([r"C:\Users\peter\anaconda3\envs\torchgpu\python.exe", "-m", "jupyter", "nbconvert", "--to", "html", "--output", f"{runName}.html", "--output-dir", "results", notebookName], check=True)

#WaitForHtml
while not htmlFile.exists(): time.sleep(0.5)

#WaitForWrite
time.sleep(2)

########################################################
# Save HTML To Same MlFlow Run
########################################################

with mlflow.start_run(run_id=runId):
    mlflow.log_artifact(str(htmlFile), artifact_path="notebook")

########################################################
# Save HTML To GitHub
########################################################

subprocess.run(["git", "add", str(htmlFile)], check=True)

hasResultChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasResultChanges:
    subprocess.run(["git", "commit", "-m", f"Results for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Tag Experiment
########################################################

existingTags = subprocess.check_output(["git", "tag"], text=True).splitlines()

if runName not in existingTags:
    subprocess.run(["git", "tag", "-a", runName, "-m", runName], check=True)
    subprocess.run(["git", "push", "origin", runName], check=True)

########################################################
# Print
########################################################

print("HTML =", htmlFile)
print("MlFlow run =", runName)
print("Saved to MLflow and GitHub")
#PrintTime
print("Current Time:", datetime.now().strftime("%H:%M:%S"))
